# OBJECTIVES - as to-do list

## Element 1
To provide stats of a thesis's complexity by extracting counts directly from the PDF source for the following metrics:
- [] number of figures: ``num_figures``
- [] number of tabels: ``num_tables`` 
- [] number of equations: ``num_equations``
- [] number of references: ``num_references``
- [] number of citations: ``num_citations``

## Element 2
To provide a measure of the "degree" of scientific grounding metrics ``citation_density``and ``figure_density`` will be calculated alongside an exstraction of the length (in number of pages) of the *litterature review* / *background* section(s):
- [] ``citation_density``(eg. a measure like $num_{citations}/1000 words$)
- [] ``figure_density`` (eg. a measure like $num_{figures}/1000words$)
- [] ``background_length``

## Element 3
An analysis of the linguistics of the thesis, measure could for example be (***suggestions!***):
- [] ``avg_sentence_length`` (number of words in the average sentence)
- [] ``hapax_legomenon_ratio`` (ratio of "words or a form that occurs only once within a particular corpus or body")
- [] others?

# PRODUCTION

# DEVELOPMENT

In [ ]:
# ==== IMPORTS ====
from pathlib import Path
import re
import fitz
import pandas as pd

In [ ]:
# ==== FUNCTIONS ====
def extract_num_figures(pdf_path: str, debug: bool = False) -> int | None:
    """Estimate number of figures with List-of-Figures fast-track and caption fallback."""
    try:
        token_pattern = r"(?:figure|fig\.?|figur|f\s*i\s*g(?:\s*u\s*r(?:\s*e)?)?\.?)"
        arabic_index_pattern = r"(?:[A-Z]\s*[\.-]\s*)?\d+(?:\s*[\.,-]\s*\d+)*(?:\s*[a-zA-Z])?"
        roman_index_pattern = r"[IVXLCDM]{1,7}"
        index_pattern = rf"\(?\s*(?:{arabic_index_pattern}|{roman_index_pattern})\s*\)?"

        caption_start_pattern = re.compile(
            rf"^\s*(?P<label>{token_pattern})\s*(?P<idx>{index_pattern})\s*(?P<sep>[:\-\.,])?\s*(?P<tail>.*)$",
            re.IGNORECASE,
        )
        caption_inline_strong_pattern = re.compile(
            rf"(?<!\w)(?P<label>{token_pattern})\s*(?P<idx>{index_pattern})\s*(?P<sep>[:\-])\s*(?P<tail>.+)$",
            re.IGNORECASE,
        )
        token_presence_pattern = re.compile(
            rf"(?<!\w){token_pattern}(?!\w)",
            re.IGNORECASE,
        )

        lof_heading_terms = (
            "list of figures",
            "figure list",
            "lof",
            "figures",
            "figurer",
            "figurenliste",
            "figur liste",
            "liste over figurer",
            "figuroversigt",
            "oversigt over figurer",
            "fortegnelse over figurer",
            "liste af figurer",
        )
        numbered_heading_prefix_pattern = rf"(?:{roman_index_pattern}|[A-Z]|\d+(?:\s*[\.-]\s*\d+)*)"
        lof_heading_pattern = re.compile(
            rf"^\s*(?:{numbered_heading_prefix_pattern}\s*[\).:-]?\s+)?(?:{'|'.join(re.escape(term) for term in lof_heading_terms)})\s*:?\s*$",
            re.IGNORECASE,
        )
        # Typical List-of-Figures entry: 'Figure 3.2 .... 17' or 'Figur 4-1: caption 42'
        lof_entry_pattern = re.compile(
            rf"^\s*(?P<label>{token_pattern})\s*(?P<idx>{index_pattern})\b.*$",
            re.IGNORECASE,
        )
        # Split-layout entries can appear as standalone index lines: '1', '2.1', 'A-3'.
        lof_standalone_idx_pattern = re.compile(
            rf"^\s*(?P<idx>{index_pattern})\s*$",
            re.IGNORECASE,
        )
        # LOF entries often appear as '3.10 Caption ...' without an explicit Figure token.
        lof_idx_caption_pattern = re.compile(
            rf"^\s*(?P<idx>{index_pattern})\s+(?P<tail>.+)$",
            re.IGNORECASE,
        )
        caption_letters_pattern = re.compile(r"[A-Za-zÆØÅæøå]")
        # Rough heading detector used to stop list parsing when another section starts.
        generic_heading_pattern = re.compile(
            r"^\s*(chapter\s+\d+|kapitel\s+\d+|references|litteratur|appendix|bilag)\b",
            re.IGNORECASE,
        )
        table_heading_terms = (
            "list of tables",
            "table list",
            "tables",
            "tabeller",
            "liste over tabeller",
            "fortegnelse over tabeller",
            "liste af tabeller",
            "tabeloversigt",
            "oversigt over tabeller",
        )
        table_heading_pattern = re.compile(
            rf"^\s*(?:{numbered_heading_prefix_pattern}\s*[\).:-]?\s+)?(?:{'|'.join(re.escape(term) for term in table_heading_terms)})\s*:?\s*$",
            re.IGNORECASE,
        )

        ignored_heading_markers = (
            "list of figures",
            "figure list",
            "lof",
            "figures",
            "figurer",
            "figurenliste",
            "figur liste",
            "liste over figurer",
            "figuroversigt",
            "oversigt over figurer",
            "fortegnelse over figurer",
            "liste af figurer",
        )
        in_text_cues = (
            "see",
            "shown in",
            "illustrated in",
            "as seen in",
            "som vist i",
            "se",
            "se også",
            "illustreret i",
            "consider",
            "at",
            "on",
            "again",
            ". ", # might be to agressive...
            "from",
        )
        no_sep_reference_tail_starters = (
            "for ",
            "in ",
            "of ",
            "from ",
        )

        def _normalize_idx(idx_text: str) -> str:
            normalized = re.sub(r"\s+", "", idx_text.strip("() ")).replace(",", ".")
            return normalized.upper()

        doc = fitz.open(pdf_path)
        pages_lines = []
        for page_num, page in enumerate(doc, start=1):
            page_text = page.get_text("text") or ""
            lines = [re.sub(r"\s+", " ", ln).strip() for ln in page_text.splitlines()]
            pages_lines.append((page_num, lines))

        # ---- Fast-track: count entries inside List-of-Figures section ----
        lof_mode = False
        lof_seen = False
        lof_entries = set()
        lof_first_numbers = set()
        lof_debug_lines = []
        pending_lof_idx = None
        pending_lof_meta = None
        pending_lof_first_num = None
        lof_start_page = None
        lof_max_span_pages = 5

        for page_num, lines in pages_lines:
            for line_num, line in enumerate(lines, start=1):
                if not line:
                    continue

                if lof_heading_pattern.match(line):
                    pending_lof_idx = None
                    pending_lof_meta = None
                    pending_lof_first_num = None
                    lof_mode = True
                    lof_seen = True
                    lof_start_page = page_num
                    if debug:
                        lof_debug_lines.append((page_num, line_num, "lof-heading", line))
                    continue

                if not lof_mode:
                    continue
                if lof_start_page is not None and page_num - lof_start_page > lof_max_span_pages:
                    if pending_lof_idx is not None and debug:
                        p_page, p_line = pending_lof_meta
                        lof_debug_lines.append((p_page, p_line, "lof-standalone-rejected-span", pending_lof_idx))
                    pending_lof_idx = None
                    pending_lof_meta = None
                    pending_lof_first_num = None
                    lof_mode = False
                    if debug:
                        lof_debug_lines.append((page_num, line_num, "lof-end-span", line))
                    continue
                if table_heading_pattern.match(line):
                    if pending_lof_idx is not None and debug:
                        p_page, p_line = pending_lof_meta
                        lof_debug_lines.append((p_page, p_line, "lof-standalone-rejected-table-switch", pending_lof_idx))
                    pending_lof_idx = None
                    pending_lof_meta = None
                    pending_lof_first_num = None
                    lof_mode = False
                    if debug:
                        lof_debug_lines.append((page_num, line_num, "lof-end-table-heading", line))
                    continue

                if generic_heading_pattern.match(line):
                    if pending_lof_idx is not None and debug:
                        p_page, p_line = pending_lof_meta
                        lof_debug_lines.append((p_page, p_line, "lof-standalone-rejected-no-caption", pending_lof_idx))
                    pending_lof_idx = None
                    pending_lof_meta = None
                    pending_lof_first_num = None
                    lof_mode = False
                    if debug:
                        lof_debug_lines.append((page_num, line_num, "lof-end-heading", line))
                    continue

                # If we hit many empty/non-entry lines, list may be over; handled naturally by no matches.
                entry_match = lof_entry_pattern.match(line)
                if entry_match:
                    idx_norm = _normalize_idx(entry_match.group("idx"))
                    lof_entries.add(idx_norm)
                    first_num_match = re.search(r"\d+", idx_norm)
                    if first_num_match:
                        lof_first_numbers.add(int(first_num_match.group()))
                    pending_lof_idx = None
                    pending_lof_meta = None
                    pending_lof_first_num = None
                    if debug:
                        lof_debug_lines.append((page_num, line_num, "lof-entry", line))
                    continue

                standalone_idx_match = lof_standalone_idx_pattern.match(line)
                if standalone_idx_match:
                    idx_norm = _normalize_idx(standalone_idx_match.group("idx"))
                    if pending_lof_idx is not None and debug:
                        p_page, p_line = pending_lof_meta
                        lof_debug_lines.append((p_page, p_line, "lof-standalone-replaced", pending_lof_idx))
                    pending_lof_idx = idx_norm
                    pending_lof_meta = (page_num, line_num)
                    first_num_match = re.search(r"\d+", idx_norm)
                    pending_lof_first_num = int(first_num_match.group()) if first_num_match else None
                    if debug:
                        lof_debug_lines.append((page_num, line_num, "lof-standalone-candidate", line))
                    continue

                idx_caption_match = lof_idx_caption_pattern.match(line)
                if idx_caption_match:
                    idx_norm = _normalize_idx(idx_caption_match.group("idx"))
                    tail = idx_caption_match.group("tail").strip()
                    if caption_letters_pattern.search(tail) and not table_heading_pattern.match(tail):
                        lof_entries.add(idx_norm)
                        first_num_match = re.search(r"\d+", idx_norm)
                        if first_num_match:
                            lof_first_numbers.add(int(first_num_match.group()))
                        if debug:
                            lof_debug_lines.append((page_num, line_num, "lof-leading-idx-entry", line))
                        pending_lof_idx = None
                        pending_lof_meta = None
                        pending_lof_first_num = None
                        continue

                if pending_lof_idx is not None and caption_letters_pattern.search(line):
                    caption_word_count = len(line.split())
                    is_heading_like = bool(lof_heading_pattern.match(line) or generic_heading_pattern.match(line))
                    has_table_switch = bool(table_heading_pattern.match(line))
                    max_seen_first = max(lof_first_numbers) if lof_first_numbers else None
                    is_plausible_first_num = (
                        pending_lof_first_num is not None
                        and pending_lof_first_num <= 60
                        and (
                            max_seen_first is None
                            or pending_lof_first_num <= max_seen_first + 5
                            or pending_lof_first_num in lof_first_numbers
                        )
                    )
                    if is_heading_like or has_table_switch or caption_word_count < 1 or not is_plausible_first_num:
                        if debug:
                            lof_debug_lines.append((page_num, line_num, "lof-standalone-rejected-caption-check", line))
                        continue
                    lof_entries.add(pending_lof_idx)
                    lof_first_numbers.add(pending_lof_first_num)
                    if debug:
                        p_page, p_line = pending_lof_meta
                        lof_debug_lines.append((p_page, p_line, "lof-standalone-accepted", pending_lof_idx))
                        lof_debug_lines.append((page_num, line_num, "lof-caption-for-standalone", line))
                    pending_lof_idx = None
                    pending_lof_meta = None
                    pending_lof_first_num = None
                    continue

                if debug and token_presence_pattern.search(line):
                    lof_debug_lines.append((page_num, line_num, "lof-non-entry", line))
        if pending_lof_idx is not None and debug:
            p_page, p_line = pending_lof_meta
            lof_debug_lines.append((p_page, p_line, "lof-standalone-rejected-eof", pending_lof_idx))

        if lof_seen and lof_entries:
            if debug:
                print(f"[DEBUG num_figures] {pdf_path}")
                print("  mode=fast-track-list-of-figures")
                print(f"  lof_entries={len(lof_entries)}")
                for p, l, reason, text in lof_debug_lines[:120]:
                    print(f"  + p{p}:l{l} [{reason}] {text}")
                if len(lof_debug_lines) > 120:
                    print(f"  ... {len(lof_debug_lines) - 120} more LOF lines omitted")
            doc.close()
            return len(lof_entries)

        # ---- Fallback: existing caption-based detection ----
        unique_keys = set()
        accepted_debug = []
        rejected_debug = []

        for page_num, lines in pages_lines:
            for line_num, line in enumerate(lines, start=1):
                if not line:
                    continue

                lower_line = line.lower()
                has_figure_token = bool(token_presence_pattern.search(line))

                if any(marker in lower_line for marker in ignored_heading_markers):
                    if debug and has_figure_token:
                        rejected_debug.append((page_num, line_num, "heading/list section", line))
                    continue

                if len(line) > 220:
                    if debug and has_figure_token:
                        rejected_debug.append((page_num, line_num, "too long for caption", line))
                    continue

                match_start = caption_start_pattern.match(line)
                if match_start:
                    sep = (match_start.group("sep") or "").strip()
                    tail = (match_start.group("tail") or "").strip().lower()
                    if not sep and any(tail.startswith(starter) for starter in no_sep_reference_tail_starters):
                        if debug:
                            rejected_debug.append((page_num, line_num, "start-line reference fragment (no separator)", line))
                        continue
                    idx_raw = match_start.group("idx")
                    idx_compact = re.sub(r"\s+", "", idx_raw.strip("() "))
                    if (
                        not sep
                        and idx_compact
                        and idx_compact[-1].isalpha()
                        and tail
                    ):
                        split_joined_tail = (idx_compact[-1] + tail).lower()
                        if any(split_joined_tail.startswith(starter.strip()) for starter in no_sep_reference_tail_starters):
                            if debug:
                                rejected_debug.append((page_num, line_num, "start-line reference fragment (split token)", line))
                            continue
                    idx_norm = _normalize_idx(idx_raw)
                    key = (page_num, idx_norm)
                    unique_keys.add(key)
                    if debug:
                        accepted_debug.append((page_num, line_num, "start-of-line caption", line))
                    continue

                match_inline = caption_inline_strong_pattern.search(line)
                if match_inline:
                    prefix = line[: match_inline.start()].lower()
                    if any(cue in prefix for cue in in_text_cues):
                        if debug:
                            rejected_debug.append((page_num, line_num, "in-text reference cue", line))
                        continue

                    if match_inline.start() <= 20 and len(line) <= 160:
                        idx_raw = match_inline.group("idx")
                        idx_norm = _normalize_idx(idx_raw)
                        key = (page_num, idx_norm)
                        unique_keys.add(key)
                        if debug:
                            accepted_debug.append((page_num, line_num, "inline strong fallback", line))
                        continue

                    if debug:
                        rejected_debug.append((page_num, line_num, "weak inline candidate", line))
                    continue

                if debug and has_figure_token:
                    rejected_debug.append((page_num, line_num, "token seen but no valid caption format", line))

        doc.close()

        if debug:
            print(f"[DEBUG num_figures] {pdf_path}")
            print("  mode=fallback-caption-scan")
            if lof_seen:
                print("  note=List-of-Figures heading found but no parseable entries; fallback used")
            print(f"  accepted_candidates={len(accepted_debug)} unique_count={len(unique_keys)}")
            for p, l, reason, text in accepted_debug[:60]:
                print(f"  + p{p}:l{l} [{reason}] {text}")
            if len(accepted_debug) > 60:
                print(f"  ... {len(accepted_debug) - 60} more accepted lines omitted")

            print(f"  rejected_candidates={len(rejected_debug)}")
            for p, l, reason, text in rejected_debug[:80]:
                print(f"  - p{p}:l{l} [{reason}] {text}")
            if len(rejected_debug) > 80:
                print(f"  ... {len(rejected_debug) - 80} more rejected lines omitted")

        return len(unique_keys)
    except Exception as e:
        print(f"[extract_num_figures] Failed for {pdf_path}: {e}")
        return None

In [ ]:
# ==== MAIN EXECUTION ====
# Local-only test run for metric 1.1: num_figures

local_pdf_root = Path("../Data/RAW_test")

pdf_paths = sorted(local_pdf_root.rglob("*.pdf"))



# Debug controls for false-positive inspection

DEBUG_NUM_FIGURES = False # True

DEBUG_MAX_FILES = 3



### ---- LOAD PDFs ---- ###

if not pdf_paths:

    print(f"No PDFs found under: {local_pdf_root.resolve()}")

    selected_paths = []

else:

    user_choice = input(

        f"Found {len(pdf_paths)} PDFs. Enter number of files to test or 'all': "

    ).strip().lower()



    if user_choice == "all":

        selected_paths = pdf_paths

    else:

        try:

            n_files = int(user_choice)

            if n_files <= 0:

                print("Please enter a positive number or 'all'.")

                selected_paths = []

            else:

                selected_paths = pdf_paths[:n_files]

        except ValueError:

            print("Invalid input. Please enter a number or 'all'.")

            selected_paths = []



    print_outputs = input("Print outputs for each file? (y/N): ").strip().lower()



### ---- 1.1: num_figures ---- ###

num_figures_records = []



if selected_paths:

    print(f"Testing num_figures on {len(selected_paths)} PDFs")

    if DEBUG_NUM_FIGURES:

        print(f"Debug mode is ON (detailed output for first {DEBUG_MAX_FILES} files)")



    for idx, pdf_path in enumerate(selected_paths):

        try:

            debug_this_file = DEBUG_NUM_FIGURES and idx < DEBUG_MAX_FILES

            num_figures = extract_num_figures(str(pdf_path), debug=debug_this_file)

            num_figures_records.append(

                {

                    "pdf_file": pdf_path.name,

                    "num_figures": num_figures,

                }

            )

            if print_outputs == "y":

                print(f"{pdf_path.name}\n num_figures={num_figures}")

        except Exception as e:

            print(f"[MAIN EXECUTION] Failed for {pdf_path}: {e}")

            num_figures_records.append(

                {

                    "pdf_file": pdf_path.name,

                    "num_figures": None,

                }

            )



# Finalized Element 1.1 table

num_figures_df = pd.DataFrame(num_figures_records, columns=["pdf_file", "num_figures"])



print(f"\nBuilt num_figures_df with {len(num_figures_df)} rows")

display(num_figures_df)


### VALIDATION CHECK for first 12 local pdf files

In [ ]:
validation = pd.DataFrame({
    "pdf_file": [
        "5cee68eed9001d2064299318_Deep Reinforcement Learning Applying an actor-critic network to a search and rescue robotics setting (translated Deep Re.pdf", 
        "5cf3aee6d9001d2064299346_Human response to increased temperatures in offices (translated Humane respons til forøget temperatur i kontormiljø).pdf",
        "5d04d261d9001d010a45c03d_Comparativ analysis of intervention strategies for a high rise building using Fire Brigade Intervention Model indsatsstr.pdf",
        "5d1c8d6bd9001d146569a4b3_Characterising eye-motion patterns in patients with autistic spectrum disorders using machine learning (translated Karak.pdf",
        "5d1c8d79d9001d146569a4dc_Deep speech recognition in Danish (translated Dyb talegenkendelse pa dansk).pdf",
        "5d1c8d7bd9001d146569a4ec_Impact of probiotic tropodithietic acid (TDA) producing Phaeobacter piscinae on microbial community members related to a.pdf",
        "5d1c8d83d9001d146569a4fd_Storage of heat in the pipeline of existing district heating grid with heat pump supply (translated Lagring af varme i r.pdf",
        "5d1c8d92d9001d146569a522_Investigating the impact of blade desing parameters on blade stability using CFD (translated Undersøgelse af virkningen .pdf",
        "5d25c7e1d9001d2d454f05ac_Facilitating longitudinal interaction in a learning environment, using microgoals.pdf",
        "5d34488fd9001d016b24698d_Vulnerability assessment of European fisheries to climate change (translated Analyse af Europas fiskeris sarbarhed over .pdf",
        "5d3d8341d9001d32f558c139_Application-Specific Hardware to Accelerate Neural Network Training (translated Hardwareaccelerator til træning af neura.pdf",
        "5d3d836cd9001d32f558c193_Optimal Scheduling of Railway Infrastructure Maintenance to Minimize Disruption (translated Optimal Planlæggning af Jern.pdf"
                 ],
    "num_figures": [
        20,24,40,46,None,12,62,38,13,11,28,28
    ],
    "num_tables": [12,26,14,4,None,5,9,9,3,12,13,14],
    "num_equations": [0,0,0,0,0,0,0,0,0,0,0,0], # NOT DATA ATM.
    "num_references": [0,0,0,0,0,0,0,0,0,0,0,0], # NOT DATA ATM.
    "num_citations": [0,0,0,0,0,0,0,0,0,0,0,0] # NOT DATA ATM.
})

#display(validation[["pdf_file", "num_figures"]])

In [ ]:
# ==== VALIDATION EVALUATION: num_figures vs true counts ====
from pathlib import Path
import pandas as pd

validation_eval = validation.copy()
validation_eval = validation_eval[validation_eval["num_figures"].notna()].copy()

raw_root = Path("../Data/RAW_test")
if not raw_root.exists():
    raw_root = Path("Data/RAW_test")

predictions = []
missing_files = []

for _, row in validation_eval.iterrows():
    file_name = str(row["pdf_file"]).strip()
    pdf_path = raw_root / file_name
    if not pdf_path.exists():
        missing_files.append(file_name)
        predictions.append(None)
        continue

    pred = extract_num_figures(str(pdf_path), debug=False)
    predictions.append(pred)

validation_eval["pred_num_figures"] = predictions
validation_eval["abs_error"] = (validation_eval["pred_num_figures"] - validation_eval["num_figures"]).abs()
validation_eval["signed_error"] = validation_eval["pred_num_figures"] - validation_eval["num_figures"]

# Tolerance metrics for quick quality checks
validation_eval["within_1"] = validation_eval["abs_error"] <= 1
validation_eval["within_2"] = validation_eval["abs_error"] <= 2
validation_eval["within_10pct"] = validation_eval["abs_error"] <= (0.1 * validation_eval["num_figures"])

n_total = len(validation_eval)
n_scored = int(validation_eval["pred_num_figures"].notna().sum())
mae = validation_eval["abs_error"].dropna().mean() if n_scored else None
mape = (
    (validation_eval["abs_error"] / validation_eval["num_figures"]).replace([float("inf")], pd.NA).dropna().mean() * 100
    if n_scored else None
)
hit_within_1 = int(validation_eval["within_1"].sum()) if n_scored else 0
hit_within_2 = int(validation_eval["within_2"].sum()) if n_scored else 0
hit_within_10pct = int(validation_eval["within_10pct"].sum()) if n_scored else 0

print("Validation summary (num_figures)")
print(f"- files in validation (non-null true counts): {n_total}")
print(f"- files scored: {n_scored}")
print(f"- MAE: {mae:.2f}" if mae is not None else "- MAE: N/A")
print(f"- MAPE: {mape:.2f}%" if mape is not None else "- MAPE: N/A")
print(f"- within ±1: {hit_within_1}/{n_scored}")
print(f"- within ±2: {hit_within_2}/{n_scored}")
print(f"- within ±10% of true value: {hit_within_10pct}/{n_scored}")

if missing_files:
    print("\nMissing files (not scored):")
    for f in missing_files:
        print(f"- {f}")

display(
    validation_eval[[
        "pdf_file",
        "num_figures",
        "pred_num_figures",
        "signed_error",
        "abs_error",
        "within_1",
        "within_2",
        "within_10pct",
    ]].sort_values("abs_error", ascending=False)
)

# ARCHIVE

### QUICK PRINTS

In [ ]:
# ==== QUICK TEST CELL ====
### ---- PRINT X PAGES OF A SINGLE PDF ---- ###

import pdfplumber

pdf_input = "Data/RAW_test/5d1c8d7bd9001d146569a4ec_Impact of probiotic tropodithietic acid (TDA) producing Phaeobacter piscinae on microbial community members related to a.pdf"

# Use existing variables from the notebook
pdf_file = Path(pdf_input) if "pdf_input" in globals() else Path(pdf_path) / pdf_file_name

# Optional fallback if relative path differs by working directory
if not pdf_file.exists():
    alt = Path("../") / pdf_file
    if alt.exists():
        pdf_file = alt

x_pages = int(input("How many pages to print? ").strip())

with pdfplumber.open(str(pdf_file)) as pdf:
    total_pages = len(pdf.pages)
    n = min(x_pages, total_pages)
    print(f"Printing {n}/{total_pages} page(s) from: {pdf_file.name}\n")

    for i in range(n):
        text = pdf.pages[i].extract_text() or "[No extractable text]"
        print(f"{'=' * 20} PAGE {i + 1} {'=' * 20}")
        print(text)
        print()

In [ ]:
# ==== QUICK TEST CELL ====
### ---- PRINT PAGE X OF A SINGLE PDF ---- ###

pdf_input = "Data/RAW_test/5d1c8d7bd9001d146569a4ec_Impact of probiotic tropodithietic acid (TDA) producing Phaeobacter piscinae on microbial community members related to a.pdf"

# Keep current pdf_file logic if needed
pdf_file = Path(pdf_input) if "pdf_input" in globals() else Path(pdf_path) / pdf_file_name

if not pdf_file.exists():
    alt = Path("../") / pdf_file
    if alt.exists():
        pdf_file = alt

page_to_print = int(input("Which page to print? (1-based): ").strip())

with pdfplumber.open(str(pdf_file)) as pdf:
    total_pages = len(pdf.pages)

    if not (1 <= page_to_print <= total_pages):
        print(f"Invalid page number. Enter a value between 1 and {total_pages}.")
    else:
        i = page_to_print - 1  # convert to 0-based index
        text = pdf.pages[i].extract_text() or "[No extractable text]"
        print(f"Printing page {page_to_print}/{total_pages} from: {pdf_file.name}\n")
        print(f"{'=' * 20} PAGE {page_to_print} {'=' * 20}")
        print(text)


### TARGETED VALIDATION CHECK

In [ ]:
# ==== ARCHIVE: TARGETED VALIDATION CHECK (num_figures) ====

# HOW TO USE

# 1) Set TARGET_FILE to a filename present in validation['pdf_file'].

# 2) Ensure Cell 8 (validation table) and Cell 5 (extract_num_figures) are already run.

# 3) Run this cell.

#

# HOW TO READ

# - true_num_figures: expected value from validation table.

# - pred_num_figures: model prediction from extract_num_figures.

# - abs_error: absolute difference.

# - pct_error: absolute percent error relative to true value.

# - within_10pct: True means pass under the current tolerance rule.



from pathlib import Path



# Change this filename when doing a one-file diagnostic.

TARGET_FILE = "5d1c8d6bd9001d146569a4b3_Characterising eye-motion patterns in patients with autistic spectrum disorders using machine learning (translated Karak.pdf"



true_num_figures = float(

    validation.loc[validation["pdf_file"] == TARGET_FILE, "num_figures"].iloc[0]

)



raw_root = Path("../Data/RAW_test")

if not raw_root.exists():

    raw_root = Path("Data/RAW_test")



target_pdf = raw_root / TARGET_FILE

pred_num_figures = extract_num_figures(str(target_pdf), debug=False)



abs_error = abs(pred_num_figures - true_num_figures)

pct_error = (abs_error / true_num_figures) * 100



print("target:", target_pdf.name)

print("true_num_figures:", true_num_figures)

print("pred_num_figures:", pred_num_figures)

print("abs_error:", abs_error)

print("pct_error:", round(pct_error, 2), "%")

print("within_10pct:", pct_error <= 10)
